# Class 2 — Secrets: don't leak

A secret is configuration that becomes dangerous the moment it leaks. We look at the three doors a key escapes through — **logs, prompts, git** — and fix each one.

No server needed for this class: the leaks live in code, so we make them right here and watch them happen.

## Config vs secret — same mechanism, different handling

| | example | can you show it? |
|---|---|---|
| **config** | log level, model name, timeout | yes — safe to log/commit |
| **secret** | API keys, tokens, passwords | **never** — not in logs, prompts, git, or images |

Both arrive as environment variables. A secret just needs strict handling.

## The right way: a `.env` file, and a `.gitignore` that protects it

`KEY` below is a fake secret so we can watch it leak (and stop leaking).

In [ ]:
import os

KEY = "sk-live-DEMO123"          # the fake secret used throughout this class
demo = "_secrets_demo"
os.makedirs(demo, exist_ok=True)
with open(os.path.join(demo, ".env"), "w") as f:
    f.write(f"LOG_LEVEL=INFO\nOPENROUTER_API_KEY={KEY}\n")
with open(os.path.join(demo, ".gitignore"), "w") as f:
    f.write(".env\n")

print("files:", sorted(os.listdir(demo)))
print("\n.env  (the SECRET lives here, not in your code):")
print(open(os.path.join(demo, ".env")).read())
print(".gitignore  (so .env is never committed):")
print(open(os.path.join(demo, ".gitignore")).read())


In [ ]:
import os

env = {}
for line in open(os.path.join(demo, ".env")):
    if "=" in line:
        k, v = line.strip().split("=", 1)
        env[k] = v

LOG_LEVEL = env.get("LOG_LEVEL", "INFO")            # config: safe to expose
OPENROUTER_API_KEY = env.get("OPENROUTER_API_KEY")  # secret: protect

print("config  LOG_LEVEL          =", LOG_LEVEL, "  <- safe")
print("secret  OPENROUTER_API_KEY =", OPENROUTER_API_KEY, "  <- never log or commit")


## Leak 1 — logs

First, the mistake: log the whole request object (the key is inside).

In [ ]:
import logging, io

request = {"route": "/v1/analyze", "api_key": KEY, "question": "hi"}

buf = io.StringIO()
log = logging.getLogger("leak1")
log.handlers.clear()
log.addHandler(logging.StreamHandler(buf))
log.setLevel("INFO")

log.info(f"got request {request}")       # BAD: the whole object
print(buf.getvalue().strip())
print("key in the log? ->", KEY in buf.getvalue())


Now the fix: log only selected, safe fields.

In [ ]:
buf = io.StringIO()
log.handlers.clear()
log.addHandler(logging.StreamHandler(buf))

request_id = "req-42"
log.info(f"request_id={request_id} route={request['route']} status=200")   # GOOD
print(buf.getvalue().strip())
print("key in the log? ->", KEY in buf.getvalue())


## Leak 2 — prompts

The user pastes their key into a support message. First, the mistake: shove the raw input into the prompt.

In [ ]:
user_message = f"I keep getting 401. My key is {KEY} can you check?"

leaky_prompt = f"You are a support agent. Answer this ticket:\n{user_message}"
print(leaky_prompt)
print("\nkey sent to the model? ->", KEY in leaky_prompt)


Now the fix: redact before the text reaches the model.

In [ ]:
import re

def redact(text):
    return re.sub(r"sk-[A-Za-z0-9-]+", "[REDACTED]", text)

safe_prompt = f"You are a support agent. Answer this ticket:\n{redact(user_message)}"
print(safe_prompt)
print("\nkey sent to the model? ->", KEY in safe_prompt)


## Leak 3 — git

The scary one. Commit a `.env`, then delete it and add it to `.gitignore`. Watch what a fresh `git log` can still recover.

In [ ]:
import subprocess, sys
out = subprocess.run([sys.executable, "git_leak_demo.py"], capture_output=True, text=True)
print(out.stdout)


### The fix

Deleting the file and adding `.gitignore` only stops *future* commits. A key that was ever committed is still in history.

1. **Rotate the key** — this is the real fix. Assume it is compromised and issue a new one.
2. Then clean up history if the repo is shared: `git filter-repo --invert-paths --path .env` (rewrites every commit).
3. Prevent it next time: add `.env` to `.gitignore` **before** your first commit, and never `git add .env`.

## The honest reality

* **Rotation beats scrubbing.** If a key ever touched git, history, or a log, treat it as leaked.
* In production, secrets do not live in a file on a laptop — they live in the cloud platform's **secret manager**, injected at runtime. (We will see exactly that in the deploy class.)
* The provider key (your OpenRouter key) is the dangerous one: your server must *send* it, so it has to exist somewhere. Keep it in the environment, never in code.

## Recap

> A secret has three doors — logs, prompts, git. Redact before you log, redact before you prompt, and never let it touch a commit. If it does, rotate it.

## What to try

* Add a config value (`LOG_LEVEL=DEBUG`) and confirm it is safe to log, unlike the key.
* Improve `redact()` to also catch a bearer token (`Bearer ...`).
* In the git demo, add `.env` to `.gitignore` *before* the first commit and confirm `git log --all -p -- .env` returns nothing.